# Arabic Sentiment Analysis — NLP Pipeline

**Goal:** Classify Arabic tweets as **positive** or **negative** using TF-IDF features and three classical ML classifiers.

| File | Contents |
|------|----------|
| `train.tsv` | ~28 k negative samples |
| `test.tsv`  | ~28 k positive samples |

> The two files are combined and re-split 80 / 20 with stratification.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import re, os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline

## 1. Load Data

In [ ]:
neg_df = pd.read_csv('train.tsv', sep='\t', header=None, names=['label', 'text'])
pos_df = pd.read_csv('test.tsv',  sep='\t', header=None, names=['label', 'text'])

df = pd.concat([neg_df, pos_df], ignore_index=True).sample(frac=1, random_state=42)
df = df.dropna(subset=['text'])

print(f'Total samples : {len(df):,}')
display(df['label'].value_counts().rename('count').to_frame())
display(df.head(5))

## 2. Text Preprocessing

Key steps:
1. Map common emojis → sentiment tokens (preserve signal)
2. Remove URLs, mentions, hashtag symbols
3. Strip Arabic diacritics (tashkeel)
4. Normalise Alef variants, Ya, Ta-marbuta
5. Remove non-Arabic characters

In [ ]:
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u065F]')
WHITESPACE        = re.compile(r'\s+')

EMOJI_SENTIMENT = {
    '😍': ' emoji_love ', '😊': ' emoji_happy ', '😂': ' emoji_laugh ',
    '❤':  ' emoji_love ', '💙': ' emoji_love ',  '🥰': ' emoji_love ',
    '😢': ' emoji_sad ',  '😭': ' emoji_cry ',   '💔': ' emoji_broken_heart ',
    '😞': ' emoji_sad ',  '😡': ' emoji_angry ', '😔': ' emoji_sad ',
    '🙂': ' emoji_happy ','😄': ' emoji_happy ', '🎉': ' emoji_pos ',
    '👏': ' emoji_pos ',  '🌚': ' emoji_dark ',  '😠': ' emoji_angry ',
}

def preprocess(text: str) -> str:
    if not isinstance(text, str): return ''
    for emoji, token in EMOJI_SENTIMENT.items():
        text = text.replace(emoji, token)
    text = re.sub(r'http\S+|www\S+', '', text)       # URLs
    text = re.sub(r'@\w+', '', text)                  # mentions
    text = text.replace('#', '')                       # hashtag symbol
    text = ARABIC_DIACRITICS.sub('', text)             # tashkeel
    text = re.sub(r'[إأآا]', 'ا', text)               # Alef normalise
    text = re.sub(r'ى',      'ي', text)               # Alef maqsura
    text = re.sub(r'ة',      'ه', text)               # Ta marbuta
    text = re.sub(r'[^\u0600-\u06FF\s_a-z]', ' ', text)  # keep Arabic + tokens
    return WHITESPACE.sub(' ', text).strip()

df['clean_text'] = df['text'].apply(preprocess)
df = df[df['clean_text'].str.len() > 2]
print(f'Samples after cleaning: {len(df):,}')

# Show example
idx = df[df['label'] == 'neg'].index[5]
print('\nOriginal  :', df.loc[idx, 'text'])
print('Processed :', df.loc[idx, 'clean_text'])

## 3. Exploratory Data Analysis

In [ ]:
PALETTE = {'pos': '#4CAF50', 'neg': '#E53935'}
df['text_length'] = df['clean_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[PALETTE[l] for l in counts.index], width=0.5)
for i, (label, v) in enumerate(counts.items()):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Label Distribution'); axes[0].set_ylabel('Count')

# Text length
for lbl, color in PALETTE.items():
    subset = df[df['label'] == lbl]['text_length']
    axes[1].hist(subset, bins=50, alpha=0.65, color=color,
                 label=f'{lbl} (mean={subset.mean():.1f})', density=True)
axes[1].set_xlim(0, 80); axes[1].set_xlabel('Word Count')
axes[1].set_title('Text Length Distribution'); axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Feature Extraction — TF-IDF Character N-grams

Character-level n-grams (2–5) outperform word n-grams on Arabic because they handle:
- Clitics and morphological affixes
- Dialect spelling variations
- Out-of-vocabulary words

In [ ]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(
    analyzer='char_wb',   # word-boundary character n-grams
    ngram_range=(2, 5),
    max_features=80_000,
    sublinear_tf=True,
    min_df=2,
)

print(f'Train : {len(X_train):,}   Test : {len(X_test):,}')

## 5. Train & Evaluate Three Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000),
    'Naive Bayes':         MultinomialNB(alpha=0.1),
    'Linear SVM':          LinearSVC(C=1.0, max_iter=2000),
}

results   = {}
pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([('tfidf', tfidf), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    try:
        scores = pipe.decision_function(X_test)
    except AttributeError:
        scores = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score((y_test == 'pos').astype(int), scores)
    results[name]   = dict(accuracy=acc, auc=auc, y_pred=y_pred, scores=scores)
    pipelines[name] = pipe
    print(f'{name:<22}  acc={acc:.4f}   auc={auc:.4f}')

## 6. Detailed Results — Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]['accuracy'])
best      = results[best_name]
print(f'Best model: {best_name}\n')
print(classification_report(y_test, best['y_pred'],
                             target_names=['neg','pos'], digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, best['y_pred'], labels=['neg','pos'])
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['neg','pos'], yticklabels=['neg','pos'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix — {best_name}')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

# ROC curve
fpr, tpr, _ = roc_curve((y_test == 'pos').astype(int), best['scores'])
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {best['auc']:.4f}")
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title(f'ROC Curve — {best_name}'); axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Model Comparison

In [ ]:
names = list(results.keys())
accs  = [results[n]['accuracy'] for n in names]
aucs  = [results[n]['auc']      for n in names]
x     = np.arange(len(names)); w = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, accs, w, label='Accuracy', color='#1565C0', alpha=0.85)
ax.bar(x + w/2, aucs, w, label='AUC-ROC',  color='#F57F17', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylim(0.80, 1.0); ax.set_ylabel('Score')
ax.set_title('Model Comparison'); ax.legend()
plt.tight_layout(); plt.show()

## 8. Predict on New Text

In [ ]:
def predict_sentiment(texts, model_name=None):
    """Predict sentiment for Arabic text."""
    if isinstance(texts, str): texts = [texts]
    pipe    = pipelines[model_name or best_name]
    cleaned = [preprocess(t) for t in texts]
    preds   = pipe.predict(cleaned)
    try:
        scores = pipe.decision_function(cleaned)
        confs  = 1 / (1 + np.exp(-np.abs(scores)))
    except AttributeError:
        confs = pipe.predict_proba(cleaned).max(axis=1)
    return [
        {'text': t, 'label': p, 'confidence': round(float(c), 4)}
        for t, p, c in zip(texts, preds, confs)
    ]

# ── Try it yourself ───────────────────────────────────────────────────────────
demo = [
    'الله يستر والله مو قادرة اتخيل',           # neg
    'الحمدلله يوم جميل وقلب سعيد',              # pos
    'مبروك عليك هذا النجاح يا صديقي',           # pos
    'والله زعلت منه كثير ومو راضي',             # neg
]

for r in predict_sentiment(demo):
    icon = '✅' if r['label'] == 'pos' else '❌'
    print(f"{icon} [{r['label'].upper()} {r['confidence']:.0%}]  {r['text']}")